In [1]:
import argparse
import torch
import os
import json
from pilgrim import TrainerValue, PilgrimValue
from pilgrim import count_parameters, generate_inverse_moves, load_permutation_data

In [2]:
# for permutation_size in range(18,25):
# # Set the arguments directly in the notebook
#     args = argparse.Namespace(
#         epochs=300,  # Number of training epochs
#         batch_size=10000,  # Batch size
#         lr=0.001,  # Learning rate
#         dropout=0.0,  # Dropout
#         optimizer="Adam",  # Optimizer (Adam or AdamSF)
#         activation="relu",  # Activation function (relu or mish)
#         use_batch_norm=True,  # Batch normalization usage (True or False, default True)
#         K_min=1,  # Minimum K value for random walks
#         K_max=50,  # Maximum K value for random walks
#         weights='',  # Path to file with model weights
#         device_id=0,  # Device ID
#         alpha=1,  # TD-learning parameter, avg 1/α steps
#         permutation = 'standard',
#         permutation_size=permutation_size,  # Cube size (e.g., 4 for 4x4x4 cube)
#         permutation_type="all",  # Cube type (qtm or all)
#         hd1=2000,  # Size of the first hidden layer
#         hd2=1000,  # Size of the second hidden layer (0 means no second layer)
#         nrd=2  # Number of residual blocks (0 means no residual blocks)
#     )

#     # Set device (GPU if available, otherwise CPU)
#     device = torch.device("cuda" if torch.cuda.is_available() else "cpu", args.device_id)
#     print(f"Start training with {device}.")

#     # Load cube data (moves and names)
#     all_moves, move_names = load_permutation_data(args.permutation, args.permutation_size, args.permutation_type, device)

#     # Derive important cube parameters from the loaded data
#     n_gens = all_moves.size(0)  # Number of moves
#     state_size = all_moves.size(1)  # Size of the state representation
#     face_size = state_size // 6  # Size of one face of the cube

#     # Generate inverse moves
#     inverse_moves = torch.tensor(generate_inverse_moves(move_names), dtype=torch.int64, device=device)
#     if args.permutation == 'standard':
#         V0 = torch.arange(args.permutation_size, dtype=torch.int8, device=device)
#     elif args.permutation == 'cube':
#         V0 = torch.arange(6, dtype=torch.int8, device=device).repeat_interleave(face_size)
#     else:
#         raise ValueError(f"Invalid permutation: {args.permutation}")

#     # Infer model mode based on hd1, hd2, and nrd
#     if args.hd2 == 0 and args.nrd == 0:
#         mode = "MLP1"
#     elif args.hd2 > 0 and args.nrd == 0:
#         mode = "MLP2"
#     elif args.hd2 > 0 and args.nrd > 0:
#         mode = "MLP2RB"
#     else:
#         raise ValueError("Invalid combination of hd1, hd2, and nrd.")

#     # Initialize the Pilgrim model
#     model = PilgrimValue(
#         state_size=state_size,
#         hd1=args.hd1,
#         hd2=args.hd2,
#         nrd=args.nrd,
#         dropout_rate=args.dropout,
#         activation_function=args.activation
#     ).to(device)

#     if len(args.weights) > 0:
#         model.load_state_dict(torch.load(f"weights/{args.weights}.pth", weights_only=True, map_location=device))
#         print(f"Weights {args.weights} loaded.")

#     # Calculate the number of model parameters
#     num_parameters = count_parameters(model)
#     param_million = round(num_parameters / 1_000_000)  # Get the number of parameters in millions

#     # Create the training name based on mode, hidden layers, residual blocks, and number of parameters
#     name = f"{args.permutation}_{args.permutation_size}_{args.permutation_type}_{mode}_{param_million:02d}M"
#     print('name', name)

#     # Create the trainer
#     trainer = TrainerValue(
#         net=model,
#         num_epochs=args.epochs,
#         device=device,
#         batch_size=args.batch_size,
#         lr=args.lr,
#         name=name,
#         K_min=args.K_min,
#         K_max=args.K_max,
#         all_moves=all_moves,
#         inverse_moves=inverse_moves,
#         V0=V0,
#         α=args.alpha
#     )

#     # Save the arguments to a log file
#     log_dir = "logs"
#     os.makedirs(log_dir, exist_ok=True)
#     args_dict = vars(args)  # Convert the args object to a dictionary
#     args_dict["model_name"] = name
#     args_dict["model_mode"] = mode
#     args_dict["model_id"] = trainer.id
#     args_dict["num_parameters"] = num_parameters

#     # Save the args and model information in JSON format
#     with open(f"{log_dir}/model_{name}_{trainer.id}.json", "w") as f:
#         json.dump(args_dict, f, indent=4)

#     # Start the training process
#     trainer.run()

In [3]:
############## Test the model ###############

In [4]:
# import numpy as np
# import argparse
# import torch
# import os
# import json
# import time
# from pilgrim import PilgrimValue, Searcher
# from pilgrim import count_parameters, generate_inverse_moves, load_permutation_data_test

# # def scramble_given_state(list_generators, n_scrambles,    state_to_scramble  = '01234...'   ):
# #     if state_to_scramble == '01234...':
# #         state_size = len( list_generators[0] )
# #         state_to_scramble = np.arange(state_size)  
# #     state_current = state_to_scramble
# #     if isinstance(state_current,list): 
# #         state_current = np.asarray(state_current)
# #     elif isinstance(state_current,range): 
# #         state_current = np.asarray(state_current)
# #     n_gens = len(list_generators)
# #     for k in range(n_scrambles):
# #         IX_move = np.random.randint(0, n_gens, dtype = int) # random moves indixes
# #         state_current = state_current[ list_generators[IX_move]] # all_moves[IX_moves,:] ] 
# #     return state_current

# def scramble_given_state_custom(list_generators, n_scrambles, state_to_scramble='01234...'):
#     import numpy as np
#     import torch  # Добавляем импорт PyTorch

#     # Если начальное состояние задано по умолчанию, создаём массив из последовательных чисел
#     if state_to_scramble == '01234...':
#         state_size = len(list_generators[0])
#         state_to_scramble = torch.arange(state_size)  # Используем тензор вместо массива

#     # Приведение к тензору для упрощения операций
#     if isinstance(state_to_scramble, (list, range)): 
#         state_to_scramble = torch.tensor(state_to_scramble)

#     state_current = np.asarray(state_to_scramble.cpu())
#     n = len(state_current)
    
#     # Заранее выпишем саму перестановку p:
#     # p[i] = индекс в state_current, из которого возьмётся элемент на позицию i после перестановки
#     # Сначала p = [0, 1, 2, ..., n-1], а потом сделаем нужные перестановки:
#     p = np.arange(n)
#     # Обмен (1,2)
#     p[1], p[2] = p[2], p[1]
#     # Обмен (3, n-1)
#     p[3], p[n-1] = p[n-1], p[3]
#     # Обмен (4, n-2)
#     p[4], p[n-2] = p[n-2], p[4]
#     # Обмен (5, n-3)
#     p[5], p[n-3] = p[n-3], p[5]
    
#     # Теперь применим эту перестановку p n_scrambles раз
#     for _ in range(n_scrambles):
#         state_current = state_current[p]

#     return torch.tensor(state_current, device=device)



# dict_results = {}
# for permutation_size in range(18,25):
#     for file in os.listdir('weights_value'):
#         if f'standard_{permutation_size}_all_MLP2RB_06M' in file and 'e00300final' in file:
#             weights_file = file

#     args_test = argparse.Namespace(
#         permutation = 'standard',
#     permutation_size=permutation_size,  # Cube size (e.g., 4 for 4x4x4 cube)
#     permutation_type="all",  # Cube type (qtm or all)
#     tests="",
#     weights='weights_value/' + weights_file,
#     B=2**18,
#     num_attempts=1,
#     num_steps=1000,
#     tests_num=1,
#     device_id=0,
#     verbose=0,
#     shift=0,
#     return_tree=0
# )

#     log_dir = "logs"
#     forest_dir = "forest"

#     # Load model info
#     with open(f"{log_dir}/model_{'_'.join(args_test.weights.split('/')[-1].split('_')[:-1])}.json", "r") as json_file:
#         info = json.load(json_file)

#     # Load cube data (moves and names)
#     all_moves, move_names = load_permutation_data_test(args_test.permutation, args_test.permutation_size, args_test.permutation_type, device)

#     # Derive important cube parameters from the loaded data
#     n_gens = all_moves.size(0)  # Number of moves
#     state_size = all_moves.size(1)  # Size of the state representation
#     face_size = state_size // 6  # Size of one face of the cube

#     # Generate inverse moves
#     # inverse_moves = torch.tensor(generate_inverse_moves(move_names), dtype=torch.int64, device=device)
#     if args_test.permutation == 'standard':
#         V0 = torch.arange(args_test.permutation_size, dtype=torch.int8, device=device)
#     elif args_test.permutation == 'cube':
#         V0 = torch.arange(6, dtype=torch.int8, device=device).repeat_interleave(face_size)
#     else:
#         raise ValueError(f"Invalid permutation: {args_test.permutation}")


#     # Load model and weights
#     model = PilgrimValue(state_size=state_size, 
#                     hd1=info['hd1'], hd2=info['hd2'], nrd=info['nrd'], 
#                     activation_function=info.get('activation', 'relu'), 
#                     use_batch_norm=info.get('use_batch_norm', True))
#     model.load_state_dict(torch.load(args_test.weights, weights_only=False, map_location=device))
#     model.eval()

#     state_start = V0
#     n_scrambles_starting_state = 1
#     state_start = scramble_given_state_custom(all_moves, n_scrambles_starting_state, state_start ) 
#     tests = [state_start]

#     # Initialize Searcher object
#     searcher = Searcher(model=model, all_moves=all_moves, V0=V0, device=device, verbose=args_test.verbose)

#     # Extract epoch information from weights file name
#     epoch = args_test.weights.split('_')[-1][:-4]

#     # Prepare log file
#     os.makedirs(log_dir, exist_ok=True)
#     log_file_add = ""
#     if len(args_test.tests) > 0:
#         log_file_add = log_file_add + "tests_" + args_test.tests.split("/")[1].split(".")[0]
#     if args_test.shift > 0:
#         log_file_add = log_file_add + f"_shift{args_test.shift}"
#     if len(log_file_add) > 0:
#         log_file_add = "_" + log_file_add
#     log_file = f"{log_dir}/test_{info['model_name']}_{info['model_id']}_{epoch}_B{args_test.B}{log_file_add}.json"


#     results = []
#     total_length = 0
#     t1 = time.time()
#     for i, state in enumerate(tests, start=0):
#         solution_time_start = time.time()
#         global test_state
#         test_state = state  
#         global test_B
#         test_B = args_test.B
#         global test_num_steps
#         test_num_steps = args_test.num_steps
#         global test_num_attempts
#         test_num_attempts = args_test.num_attempts
#         global test_return_tree
#         test_return_tree = args_test.return_tree
#         result = searcher.get_solution(
#             state, B=args_test.B, 
#             num_steps=args_test.num_steps, num_attempts=args_test.num_attempts, 
#             return_tree=args_test.return_tree
#         )
#         moves, attempts = result[:2]
#         if args_test.return_tree and moves is not None:
#             tree = result[2]
#             os.makedirs(forest_dir, exist_ok=True)
#             torch.save(tree.cpu(), f"{forest_dir}/tree_{args_test.permutation}_{args_test.permutation_size}_{args_test.permutation_type}_i{i+args_test.shift:04d}_B{args_test.B:08d}_{info['model_id']}.pt")  
#             torch.save(state.cpu(), f"{forest_dir}/state_{args_test.permutation}_{args_test.permutation_size}_{args_test.permutation_type}_i{i+args_test.shift:04d}_B{args_test.B:08d}_{info['model_id']}.pt") 

#         solution_time_end = time.time()
#         timestamp = time.strftime('%Y-%m-%d %H:%M:%S', time.localtime())
        
#         if moves is not None:
#             solution_length = len(moves)
#             total_length += solution_length
            
#             result = {
#                 "test_num": i+args_test.shift,
#                 "solution_length": solution_length,
#                 "attempts": attempts + 1,
#                 "time": round(solution_time_end - solution_time_start, 2),
#                 "moves": moves.tolist()
#             }
            
#             # Print solution length for each solved cube
#             print(f"[{timestamp}] Solution {i}: Length = {solution_length}")
#         else:
#             # If no solution is found
#             result = {
#                 "test_num": i+args_test.shift,
#                 "solution_length": None,
#                 "attempts": None,
#                 "time": round(solution_time_end - solution_time_start, 2),
#                 "moves": None
#             }
#             print(f"[{timestamp}] Solution {i} not found")
        
#         results.append(result)

#         # Append new result to the log file
#         with open(log_file, 'w') as f:
#             json.dump(results, f, indent=4)

#     t2 = time.time()

#     # Calculate average solution length
#     solved_results = [r for r in results if r["solution_length"] is not None]
#     avg_length = total_length / len(solved_results) if solved_results else 0

#     # Print completion message with average solution length
#     print(f"Test completed in {(t2 - t1):.2f}s.")
#     print(f"Average solution length: {avg_length:.2f}.")
#     print(f"Solved {len(solved_results)}/{args_test.tests_num} permutations.")
#     print(f"Results saved to {log_file}.")

#     dict_results[permutation_size] = avg_length


# import pandas as pd
# df = pd.DataFrame(dict_results, index=['avg_length']).T.rename(columns = {'avg_length':'model_length'})
# df['reference_length'] = df.index*(df.index-1)/2
# df

In [6]:
# df_2 = df.copy()

In [14]:
import numpy as np
import argparse
import torch
import os
import json
import time
from pilgrim import PilgrimValue, Searcher
from pilgrim import count_parameters, generate_inverse_moves, load_permutation_data_test

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# def scramble_given_state(list_generators, n_scrambles,    state_to_scramble  = '01234...'   ):
#     if state_to_scramble == '01234...':
#         state_size = len( list_generators[0] )
#         state_to_scramble = np.arange(state_size)  
#     state_current = state_to_scramble
#     if isinstance(state_current,list): 
#         state_current = np.asarray(state_current)
#     elif isinstance(state_current,range): 
#         state_current = np.asarray(state_current)
#     n_gens = len(list_generators)
#     for k in range(n_scrambles):
#         IX_move = np.random.randint(0, n_gens, dtype = int) # random moves indixes
#         state_current = state_current[ list_generators[IX_move]] # all_moves[IX_moves,:] ] 
#     return state_current

def scramble_given_state_custom(list_generators, n_scrambles, state_to_scramble='01234...'):
    import numpy as np
    import torch  # Добавляем импорт PyTorch

    # Если начальное состояние задано по умолчанию, создаём массив из последовательных чисел
    if state_to_scramble == '01234...':
        state_size = len(list_generators[0])
        state_to_scramble = torch.arange(state_size)  # Используем тензор вместо массива

    # Приведение к тензору для упрощения операций
    if isinstance(state_to_scramble, (list, range)): 
        state_to_scramble = torch.tensor(state_to_scramble)

    state_current = np.asarray(state_to_scramble.cpu())
    n = len(state_current)
    
    # Заранее выпишем саму перестановку p:
    # p[i] = индекс в state_current, из которого возьмётся элемент на позицию i после перестановки
    # Сначала p = [0, 1, 2, ..., n-1], а потом сделаем нужные перестановки:
    p = torch.arange(n)
    # Обмен (0,1)
    p[0], p[1] = p[1].clone(), p[0].clone()
    # Обмен (2, n-1)
    p[2], p[n-1] = p[n-1].clone(), p[2].clone()
    # Обмен (3, n-2)
    p[3], p[n-2] = p[n-2].clone(), p[3].clone()
    # Обмен (4, n-3)
    p[4], p[n-3] = p[n-3].clone(), p[4].clone()
    state_current = state_current[p]


    # p = np.arange(n)
    # # Обмен (0,1)
    # p[4], p[n-3] = p[n-3], p[4]
    # # Обмен (2, n-1)
    # p[3], p[n-2] = p[n-2], p[3]
    # # Обмен (3, n-2)
    # p[2], p[n-1] = p[n-1], p[2]
    # # Обмен (4, n-3)
    # p[0], p[1] = p[1], p[0]
    # state_current = state_current[p]

   
    return torch.tensor(state_current, device=device)



dict_results = {}
for permutation_size in range(14,15):
    for file in os.listdir('weights_value'):
        if f'standard_{permutation_size}_all_MLP2RB_00M' in file and 'e00300final' in file:
            weights_file = file

    args_test = argparse.Namespace(
        permutation = 'standard',
    permutation_size=permutation_size,  # Cube size (e.g., 4 for 4x4x4 cube)
    permutation_type="all",  # Cube type (qtm or all)
    tests="",
    weights='weights_value/' + weights_file,
    B=2**18,
    num_attempts=1,
    num_steps=1000,
    tests_num=1,
    device_id=0,
    verbose=0,
    shift=0,
    return_tree=0
)

    log_dir = "logs"
    forest_dir = "forest"

    # Load model info
    with open(f"{log_dir}/model_{'_'.join(args_test.weights.split('/')[-1].split('_')[:-1])}.json", "r") as json_file:
        info = json.load(json_file)

    # Load cube data (moves and names)
    all_moves, move_names = load_permutation_data_test(args_test.permutation, args_test.permutation_size, args_test.permutation_type, device)

    # Derive important cube parameters from the loaded data
    n_gens = all_moves.size(0)  # Number of moves
    state_size = all_moves.size(1)  # Size of the state representation
    face_size = state_size // 6  # Size of one face of the cube

    # Generate inverse moves
    # inverse_moves = torch.tensor(generate_inverse_moves(move_names), dtype=torch.int64, device=device)
    if args_test.permutation == 'standard':
        V0 = torch.arange(args_test.permutation_size, dtype=torch.int8, device=device)
    elif args_test.permutation == 'cube':
        V0 = torch.arange(6, dtype=torch.int8, device=device).repeat_interleave(face_size)
    else:
        raise ValueError(f"Invalid permutation: {args_test.permutation}")


    # Load model and weights
    model = PilgrimValue(state_size=state_size, 
                    hd1=info['hd1'], hd2=info['hd2'], nrd=info['nrd'], 
                    activation_function=info.get('activation', 'relu'), 
                    use_batch_norm=info.get('use_batch_norm', True))
    model.load_state_dict(torch.load(args_test.weights, weights_only=False, map_location=device))
    model.eval()

    state_start = V0
    n_scrambles_starting_state = 1
    state_start = scramble_given_state_custom(all_moves, n_scrambles_starting_state, state_start ) 
    tests = [state_start]

    # Initialize Searcher object
    searcher = Searcher(model=model, all_moves=all_moves, V0=V0, device=device, verbose=args_test.verbose)

    # Extract epoch information from weights file name
    epoch = args_test.weights.split('_')[-1][:-4]

    # Prepare log file
    os.makedirs(log_dir, exist_ok=True)
    log_file_add = ""
    if len(args_test.tests) > 0:
        log_file_add = log_file_add + "tests_" + args_test.tests.split("/")[1].split(".")[0]
    if args_test.shift > 0:
        log_file_add = log_file_add + f"_shift{args_test.shift}"
    if len(log_file_add) > 0:
        log_file_add = "_" + log_file_add
    log_file = f"{log_dir}/test_{info['model_name']}_{info['model_id']}_{epoch}_B{args_test.B}{log_file_add}.json"


    results = []
    total_length = 0
    t1 = time.time()
    for i, state in enumerate(tests, start=0):
        solution_time_start = time.time()
        global test_state
        test_state = state  
        global test_B
        test_B = args_test.B
        global test_num_steps
        test_num_steps = args_test.num_steps
        global test_num_attempts
        test_num_attempts = args_test.num_attempts
        global test_return_tree
        test_return_tree = args_test.return_tree
        result = searcher.get_solution(
            state, B=args_test.B, 
            num_steps=args_test.num_steps, num_attempts=args_test.num_attempts, 
            return_tree=args_test.return_tree
        )
        moves, attempts = result[:2]
        if args_test.return_tree and moves is not None:
            tree = result[2]
            os.makedirs(forest_dir, exist_ok=True)
            torch.save(tree.cpu(), f"{forest_dir}/tree_{args_test.permutation}_{args_test.permutation_size}_{args_test.permutation_type}_i{i+args_test.shift:04d}_B{args_test.B:08d}_{info['model_id']}.pt")  
            torch.save(state.cpu(), f"{forest_dir}/state_{args_test.permutation}_{args_test.permutation_size}_{args_test.permutation_type}_i{i+args_test.shift:04d}_B{args_test.B:08d}_{info['model_id']}.pt") 

        solution_time_end = time.time()
        timestamp = time.strftime('%Y-%m-%d %H:%M:%S', time.localtime())
        
        if moves is not None:
            solution_length = len(moves)
            total_length += solution_length
            
            result = {
                "test_num": i+args_test.shift,
                "solution_length": solution_length,
                "attempts": attempts + 1,
                "time": round(solution_time_end - solution_time_start, 2),
                "moves": moves.tolist()
            }
            
            # Print solution length for each solved cube
            print(f"[{timestamp}] Solution {i}: Length = {solution_length}")
        else:
            # If no solution is found
            result = {
                "test_num": i+args_test.shift,
                "solution_length": None,
                "attempts": None,
                "time": round(solution_time_end - solution_time_start, 2),
                "moves": None
            }
            print(f"[{timestamp}] Solution {i} not found")
        
        results.append(result)

        # Append new result to the log file
        with open(log_file, 'w') as f:
            json.dump(results, f, indent=4)

    t2 = time.time()

    # Calculate average solution length
    solved_results = [r for r in results if r["solution_length"] is not None]
    avg_length = total_length / len(solved_results) if solved_results else 0

    # Print completion message with average solution length
    print(f"Test completed in {(t2 - t1):.2f}s.")
    print(f"Average solution length: {avg_length:.2f}.")
    print(f"Solved {len(solved_results)}/{args_test.tests_num} permutations.")
    print(f"Results saved to {log_file}.")

    dict_results[permutation_size] = avg_length


import pandas as pd
df = pd.DataFrame(dict_results, index=['avg_length']).T.rename(columns = {'avg_length':'model_length'})
df['reference_length'] = df.index*(df.index-1)/2
df

[2025-01-27 19:43:33] Solution 0: Length = 82
Test completed in 3.55s.
Average solution length: 82.00.
Solved 1/1 permutations.
Results saved to logs/test_standard_14_all_MLP2RB_00M_1737889113_e00300final_B262144.json.


,model_length,reference_length
14,82.0,91.0


In [8]:
import torch

# Размер перестановки 14
# Здесь записаны ходы, которые мы получили с помощью модели и beam-search
result = {}
result['moves_fixed'] = [1, 0, 1, 1, 1, 0, 1, 0, 1, 0, 1, 0, 1, 0, 1, 1, 1, 1, 0, 1, 0, 1, 0, 1, 0, 1, 1, 0, 1, 0, 1, 0, 1, 0, 1, 1, 1, 1, 1, 1, 0, 1, 0, 1, 0, 1, 1, 1, 1, 1, 1, 1, 1, 1, 0, 1, 1, 0, 1, 0, 1, 0, 1, 0, 1, 0, 1, 1, 1, 1, 1, 1, 0, 1, 0, 1, 0, 1, 1, 0, 1, 0]

# Здесь записаны все возможные ходы для перестановки 14
all_moves = torch.tensor([[ 1,  0,  2,  3,  4,  5,  6,  7,  8,  9, 10, 11, 12, 13],
        [13,  0,  1,  2,  3,  4,  5,  6,  7,  8,  9, 10, 11, 12]],
       device='cpu')

device = torch.device('cpu')

# Определяем собранное состояние
state_current = torch.arange(14, dtype=torch.int8, device=device)

# Далее применяем к собранному состоянию перестановку (12)(3n)(4(n-1))(5(n-2))
n = state_current.size(0)
p = torch.arange(n)
# Обмен (0,1)
p[0], p[1] = p[1].clone(), p[0].clone()
# Обмен (2, n-1)
p[2], p[n-1] = p[n-1].clone(), p[2].clone()
# Обмен (3, n-2)
p[3], p[n-2] = p[n-2].clone(), p[3].clone()
# Обмен (4, n-3)
p[4], p[n-3] = p[n-3].clone(), p[4].clone()
state_current = state_current[p]
print('Перестановка (12)(3n)(4(n-1))(5(n-2))', state_current)

# Переводим ходы в тензор
moves_tensor = torch.tensor(result['moves_fixed'], dtype=torch.int64, device=device)

# Инициализируем текущее состояние
current_state = state_current.clone()

# Применяем каждый ход в последовательности
for move in moves_tensor:
    current_state = torch.gather(current_state.unsqueeze(0), 1, all_moves[move].unsqueeze(0)).squeeze(0)

# Выводим результат
print('Итоговое состояние по ходам модели:', current_state)
print('длина пути по ходам модели:', len(result['moves_fixed']))
print('длина пути по формуле:', n*(n-1)/2)


Перестановка (12)(3n)(4(n-1))(5(n-2)) tensor([ 1,  0, 13, 12, 11,  5,  6,  7,  8,  9, 10,  4,  3,  2],
       dtype=torch.int8)
Итоговое состояние по ходам модели: tensor([ 0,  1,  2,  3,  4,  5,  6,  7,  8,  9, 10, 11, 12, 13],
       dtype=torch.int8)
длина пути по ходам модели: 82
длина пути по формуле: 91.0


In [15]:
all_moves

tensor([[ 1,  0,  2,  3,  4,  5,  6,  7,  8,  9, 10, 11, 12, 13],
        [13,  0,  1,  2,  3,  4,  5,  6,  7,  8,  9, 10, 11, 12]],
       device='cuda:0')

In [21]:
import torch

all_moves = torch.tensor([[ 1,  0,  2,  3,  4,  5,  6,  7,  8,  9, 10, 11, 12, 13],
        [13,  0,  1,  2,  3,  4,  5,  6,  7,  8,  9, 10, 11, 12]],
       device='cpu')

device = torch.device('cpu')
state_current = torch.arange(14, dtype=torch.int8, device=device)
n = state_current.size(0)
p = torch.arange(n)
# Обмен (0,1)
p[0], p[1] = p[1].clone(), p[0].clone()
# Обмен (2, n-1)
p[2], p[n-1] = p[n-1].clone(), p[2].clone()
# Обмен (3, n-2)
p[3], p[n-2] = p[n-2].clone(), p[3].clone()
# Обмен (4, n-3)
p[4], p[n-3] = p[n-3].clone(), p[4].clone()
state_current = state_current[p]
print(state_current)
# Convert moves to a tensor
moves_tensor = torch.tensor(result['moves'], dtype=torch.int64, device=device)

# Initialize the current state
current_state = state_current.clone()

# Apply each move in the sequence
for move in moves_tensor:
    current_state = torch.gather(current_state.unsqueeze(0), 1, all_moves[move].unsqueeze(0)).squeeze(0)

# current_state now contains the state after applying all moves
print(current_state)

tensor([ 1,  0, 13, 12, 11,  5,  6,  7,  8,  9, 10,  4,  3,  2],
       dtype=torch.int8)
tensor([ 0,  1,  2,  3,  4,  5,  6,  7,  8,  9, 10, 11, 12, 13],
       dtype=torch.int8)


In [18]:
state_start

tensor([ 1,  0, 13, 12, 11,  5,  6,  7,  8,  9, 10,  4,  3,  2],
       device='cuda:0', dtype=torch.int8)

In [19]:
# Convert moves to a tensor
moves_tensor = torch.tensor(result['moves'], dtype=torch.int64, device=device)

# Initialize the current state
current_state = state_start.clone()

# Apply each move in the sequence
for move in moves_tensor:
    current_state = torch.gather(current_state.unsqueeze(0), 1, all_moves[move].unsqueeze(0)).squeeze(0)

# current_state now contains the state after applying all moves
print(current_state)

tensor([ 0,  1,  2,  3,  4,  5,  6,  7,  8,  9, 10, 11, 12, 13],
       device='cuda:0', dtype=torch.int8)


In [34]:
all_moves

tensor([[ 1,  0,  2,  3,  4,  5,  6,  7,  8,  9, 10, 11, 12, 13],
        [13,  0,  1,  2,  3,  4,  5,  6,  7,  8,  9, 10, 11, 12]],
       device='cuda:0')

In [23]:
all_moves = tensor([[ 1,  0,  2,  3,  4,  5,  6,  7,  8,  9, 10, 11, 12, 13],
        [13,  0,  1,  2,  3,  4,  5,  6,  7,  8,  9, 10, 11, 12]],

p[4], p[n-3] = p[n-3], p[4]
# Обмен (2, n-1)
p[3], p[n-2] = p[n-2], p[3]
# Обмен (3, n-2)
p[2], p[n-1] = p[n-1], p[2]
# Обмен (4, n-3)
p[0], p[1] = p[1], p[0]

n =14
move_transp_0_1 = torch.arange(n)
move_transp_0_1[0], move_transp_0_1[1] = move_transp_0_1[1], move_transp_0_1[0]
move_transp_2_n1 = torch.arange(n)
move_transp_2_n1[2], move_transp_2_n1[n-1] = move_transp_2_n1[n-1], move_transp_2_n1[2]
move_transp_3_n2 = torch.arange(n)
move_transp_3_n2[3], move_transp_3_n2[n-2] = move_transp_3_n2[n-2], move_transp_3_n2[3]
move_transp_4_n3 = torch.arange(n)
move_transp_4_n3[4], move_transp_4_n3[n-3] = move_transp_4_n3[n-3], move_transp_4_n3[4]



tensor([[ 1,  0,  2,  3,  4,  5,  6,  7,  8,  9, 10, 11, 12, 13],
        [13,  0,  1,  2,  3,  4,  5,  6,  7,  8,  9, 10, 11, 12]],
       device='cuda:0')

In [32]:
n= 14
p = np.arange(n)
p[0], p[1] = p[1], p[0]

In [33]:
p

array([ 1,  0,  2,  3,  4,  5,  6,  7,  8,  9, 10, 11, 12, 13])

In [17]:
import torch

# Assuming state_start, result['moves'], and all_moves are already defined
# state_start: the initial state tensor
# result['moves']: the list of moves to apply
# all_moves: the tensor containing all possible moves
def apply_move(self, states, moves):
    moved_states = torch.empty(states.size(0), self.state_size, device=self.device, dtype=states.dtype)
    for i in range(0, states.size(0), self.batch_size):
        moved_states[i:i+self.batch_size] = torch.gather(states[i:i+self.batch_size], 1, self.all_moves[moves[i:i+self.batch_size]])
    return moved_states

# Convert moves to a tensor
moves_tensor = torch.tensor(result['moves'], dtype=torch.int64, device=state_start.device)

# Initialize the current state
current_state = state_start.clone()

# Apply each move in the sequence
for move in moves_tensor:
    current_state = torch.gather(current_state.unsqueeze(0), 1, all_moves[move].unsqueeze(0)).squeeze(0)

# current_state now contains the state after applying all moves
print(current_state)

tensor([ 0,  1,  2,  3,  4,  5,  6,  7,  8,  9, 10, 11, 12, 13],
       device='cuda:0', dtype=torch.int8)


In [14]:
n = 10
p = np.arange(n)
# Обмен (1,2)
p[0], p[1] = p[1], p[0]
# Обмен (3, n-1)
p[2], p[n-1] = p[n-1], p[2]
# Обмен (4, n-2)
p[3], p[n-2] = p[n-2], p[3]
# Обмен (5, n-3)
p[4], p[n-3] = p[n-3], p[4]

In [15]:
p

array([0, 1, 2, 3, 4, 5, 6, 7, 8, 9])

In [15]:
df['model_num_parameters'] = '600k'
df_2['model_num_parameters'] = '6M'

In [16]:
df_total =pd.concat([df[:8], df_2])

In [19]:
df_total = df_total.reset_index()


In [21]:
df_total = df_total.rename(columns = {'index':'permutation_size'})


In [22]:
df_total

,permutation_size,model_length,reference_length,model_num_parameters
0,10,56.0,45.0,600k
1,11,63.0,55.0,600k
2,12,70.0,66.0,600k
3,13,77.0,78.0,600k
4,14,84.0,91.0,600k
5,15,91.0,105.0,600k
6,16,98.0,120.0,600k
7,17,107.0,136.0,600k
8,18,112.0,153.0,6M
9,19,119.0,171.0,6M


In [7]:
import pandas as pd
df = pd.DataFrame(dict_results, index=['avg_length']).T.rename(columns = {'avg_length':'model_length'})
df['reference_length'] = df.index*(df.index-1)/2



In [8]:
df

,model_length,reference_length
10,56.0,45.0
11,63.0,55.0
12,70.0,66.0
13,77.0,78.0
14,84.0,91.0
15,91.0,105.0
16,98.0,120.0
17,107.0,136.0
18,112.0,153.0
19,140.0,171.0
